In [18]:
import pandas as pd
import numpy as np

may = pd.DataFrame({
    "outlet": ["Raffles Place", "raffles place", "RAFFLES PLACE", "Raffles Pl.",
               "Marina Bay", "marina bay", "Marina Bay ", "Marina Bay",
               "Tampines Mall", "tampines mall", "Holland Village", "Holland V"],
    "date_text": ["01/05/2025", "01/05/2025", "02/05/2025", "02/05/2025",
                  "01/05/2025", "01/05/2025", "02/05/2025", "02/05/2025",
                  "01/05/2025", "02/05/2025", "01/05/2025", "02/05/2025"],
    "daypart": ["Morning", "AM", "morning", "Midday",
                "Lunch", "midday", "Evening", "PM",
                "evening", "Morning", "AM", "Lunch"],
    "revenue_raw": ["S$612.40", " 318.75 ", "-999", "487.10",
                    "S$1,204.60", "296.55", "", "58000",
                    "S$734.20", "421.85", "S$389.90", "-999"],
    "tickets": [74, 39, 55, 58, 141, 36, 21, 112, 88, 51, np.nan, -3],
    "staff_on_shift": [3, 2, np.nan, 2, 5, 2, 1, 4, 4, 2, 2, np.nan],
    "manager": ["  Aisha Rahman ", "aisha rahman", "AISHA RAHMAN", "  Aisha Rahman",
                "Priya  Nair", "priya nair", "Priya Nair ", "Priya Nair",
                "wei ming tan", "Wei Ming Tan", "DANIEL LIM ", "Daniel Lim"],
    "notes": ["", "n/a", "N.A.", "-", "aircon down", "", "-", "", "n/a", "", "N.A.", ""],
})

may

,outlet,date_text,daypart,revenue_raw,tickets,staff_on_shift,manager,notes
0,Raffles Place,01/05/2025,Morning,S$612.40,74.0,3.0,Aisha Rahman,
1,raffles place,01/05/2025,AM,318.75,39.0,2.0,aisha rahman,n/a
2,RAFFLES PLACE,02/05/2025,morning,-999,55.0,NaN,AISHA RAHMAN,N.A.
3,Raffles Pl.,02/05/2025,Midday,487.10,58.0,2.0,Aisha Rahman,-
4,Marina Bay,01/05/2025,Lunch,"S$1,204.60",141.0,5.0,Priya Nair,aircon down
5,marina bay,01/05/2025,midday,296.55,36.0,2.0,priya nair,
6,Marina Bay,02/05/2025,Evening,,21.0,1.0,Priya Nair,-
7,Marina Bay,02/05/2025,PM,58000,112.0,4.0,Priya Nair,
8,Tampines Mall,01/05/2025,evening,S$734.20,88.0,4.0,wei ming tan,n/a
9,tampines mall,02/05/2025,Morning,421.85,51.0,2.0,Wei Ming Tan,


### Challenge 1: The Health Check

**Tasks:**
1. Run the five-move first look: `.head()`, `.shape`, `.info()`, `.dtypes`, `.describe()`.
2. Which column is missing from `.describe()` output, and why?
3. Use `.isna().sum()` to count missing values per column. Then look at `notes` with
   `.value_counts(dropna=False)` — how many values *look* empty but are not?
4. How many distinct outlets does pandas think there are? How many are there really?

<details>
<summary>💡 Hint</summary>

`.describe()` only sees numeric columns. `revenue_raw` is text, so it is invisible — which is the
single most important finding of the health check, because it means none of the impossible values in
it can be spotted yet.

In [19]:
## MY ANSWER ##
#1-Run the five-move first look: `.head()`, `.shape`, `.info()`, `.dtypes`, `.describe()`
may.head()
may.shape
may.info()
may.dtypes
may.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   outlet          12 non-null     object 
 1   date_text       12 non-null     object 
 2   daypart         12 non-null     object 
 3   revenue_raw     12 non-null     object 
 4   tickets         11 non-null     float64
 5   staff_on_shift  10 non-null     float64
 6   manager         12 non-null     object 
 7   notes           12 non-null     object 
dtypes: float64(2), object(6)
memory usage: 896.0+ bytes


,tickets,staff_on_shift
count,11.000000,10.000000
mean,61.090909,2.700000
std,41.047423,1.251666
min,-3.000000,1.000000
25%,37.500000,2.000000
50%,55.000000,2.000000
75%,81.000000,3.750000
max,141.000000,5.000000


#2-Which column is missing from `.describe()` output, and why?
***Answer*** .describe() only covers numeric columns, so all text columns are left out. The critical one is revenue_raw: it should be numeric but is stored as text (object) because of S$, commas and blanks. That means its impossible values (-999, 58000) can’t be spotted until it’s converted to a number.

#3-Use `.isna().sum()` to count missing values per column. Then look at `notes` with
   `.value_counts(dropna=False)` — how many values *look* empty but are not?

In [20]:
## MY ANSWER ##
display(may.isna().sum())
display(may["notes"].value_counts(dropna=False))

outlet            0
date_text         0
daypart           0
revenue_raw       0
tickets           1
staff_on_shift    2
manager           0
notes             0
dtype: int64

               5
n/a            2
N.A.           2
-              2
aircon down    1
Name: notes, dtype: int64

***Analysis*** The first table says notes 0 missing, the second shows 11 of those 12 notes are fake blanks. In other words, all 11 are meant to be empty, but to pandas they’re ordinary text, just like “aircon down”. So .isna() doesn’t see them as missing, and the first table says notes 0.

4. How many distinct outlets does pandas think there are? How many are there really?

In [21]:
## MY ANSWER ##
may["outlet"].nunique()
may["outlet"].value_counts()

Marina Bay         2
Raffles Place      1
raffles place      1
RAFFLES PLACE      1
Raffles Pl.        1
marina bay         1
Marina Bay         1
Tampines Mall      1
tampines mall      1
Holland Village    1
Holland V          1
Name: outlet, dtype: int64

***Analysis*** That’s 12 rows and 11 distinct strings, because only "Marina Bay" appears twice with identical spelling.
The reality: 4 cafés. Raffles Place, Marina Bay, Tampines Mall and Holland Village.
Why pandas gets it wrong: it compares text character by character. To pandas, these are all different.

### Challenge 2: Clean the Data
Tasks:

1. Convert `revenue_raw` into a numeric column called `revenue_sgd`.
2. Replace the fake blanks in `notes` (`""`, `n/a`, `N.A.`, `-`) with `NaN`.
3. Mask the impossible values: negative takings, takings above 20,000, and ticket counts of zero or
   less.
4. Fill the resulting holes in `revenue_sgd` and `tickets` with each column's median — **in the right
   order**.
5. Standardise `outlet` to the four real café names, and `daypart` to Morning / Midday / Evening.
6. Verify: no impossible values left, and `outlet` has exactly four values.

In [22]:
## MY ANSWER ##

#1-Convert `revenue_raw` into a numeric column called `revenue_sgd`.
clean = may.copy()
clean["revenue_sgd"] = pd.to_numeric(
    clean["revenue_raw"].str.replace(r"[^0-9.\-]", "", regex=True),
    errors="coerce",
)
display(clean[["revenue_raw", "revenue_sgd"]])
display(clean["revenue_sgd"].dtype) #verify if type is numeric

,revenue_raw,revenue_sgd
0,S$612.40,612.40
1,318.75,318.75
2,-999,-999.00
3,487.10,487.10
4,"S$1,204.60",1204.60
5,296.55,296.55
6,,NaN
7,58000,58000.00
8,S$734.20,734.20
9,421.85,421.85


dtype('float64')

In [23]:
#2-Replace the fake blanks in `notes` (`""`, `n/a`, `N.A.`, `-`) with `NaN`.
clean["notes"]=clean["notes"].replace(["", "n/a", "N.A.", "-"], np.nan)
display(clean)
display(clean["notes"].isna().sum()) #now isna can recognized NaN 

,outlet,date_text,daypart,revenue_raw,tickets,staff_on_shift,manager,notes,revenue_sgd
0,Raffles Place,01/05/2025,Morning,S$612.40,74.0,3.0,Aisha Rahman,NaN,612.40
1,raffles place,01/05/2025,AM,318.75,39.0,2.0,aisha rahman,NaN,318.75
2,RAFFLES PLACE,02/05/2025,morning,-999,55.0,NaN,AISHA RAHMAN,NaN,-999.00
3,Raffles Pl.,02/05/2025,Midday,487.10,58.0,2.0,Aisha Rahman,NaN,487.10
4,Marina Bay,01/05/2025,Lunch,"S$1,204.60",141.0,5.0,Priya Nair,aircon down,1204.60
5,marina bay,01/05/2025,midday,296.55,36.0,2.0,priya nair,NaN,296.55
6,Marina Bay,02/05/2025,Evening,,21.0,1.0,Priya Nair,NaN,NaN
7,Marina Bay,02/05/2025,PM,58000,112.0,4.0,Priya Nair,NaN,58000.00
8,Tampines Mall,01/05/2025,evening,S$734.20,88.0,4.0,wei ming tan,NaN,734.20
9,tampines mall,02/05/2025,Morning,421.85,51.0,2.0,Wei Ming Tan,NaN,421.85


11

In [24]:
#3 Mask the impossible values: negative takings, takings above 20,000, and ticket counts of zero or less.
clean['revenue_sgd'] = clean["revenue_sgd"].mask(
    (clean["revenue_sgd"] < 0) | (clean["revenue_sgd"] > 20000)
)
clean["tickets"] = clean["tickets"].mask(clean["tickets"] <= 0)
display(clean)

,outlet,date_text,daypart,revenue_raw,tickets,staff_on_shift,manager,notes,revenue_sgd
0,Raffles Place,01/05/2025,Morning,S$612.40,74.0,3.0,Aisha Rahman,NaN,612.40
1,raffles place,01/05/2025,AM,318.75,39.0,2.0,aisha rahman,NaN,318.75
2,RAFFLES PLACE,02/05/2025,morning,-999,55.0,NaN,AISHA RAHMAN,NaN,NaN
3,Raffles Pl.,02/05/2025,Midday,487.10,58.0,2.0,Aisha Rahman,NaN,487.10
4,Marina Bay,01/05/2025,Lunch,"S$1,204.60",141.0,5.0,Priya Nair,aircon down,1204.60
5,marina bay,01/05/2025,midday,296.55,36.0,2.0,priya nair,NaN,296.55
6,Marina Bay,02/05/2025,Evening,,21.0,1.0,Priya Nair,NaN,NaN
7,Marina Bay,02/05/2025,PM,58000,112.0,4.0,Priya Nair,NaN,NaN
8,Tampines Mall,01/05/2025,evening,S$734.20,88.0,4.0,wei ming tan,NaN,734.20
9,tampines mall,02/05/2025,Morning,421.85,51.0,2.0,Wei Ming Tan,NaN,421.85


In [25]:
#4- Fill the resulting holes in `revenue_sgd` and `tickets` with each column's median — **in the right order**.
clean = clean.fillna({
    "revenue_sgd": clean["revenue_sgd"].median(),
    "tickets": clean["tickets"].median(),
})
display(clean)

,outlet,date_text,daypart,revenue_raw,tickets,staff_on_shift,manager,notes,revenue_sgd
0,Raffles Place,01/05/2025,Morning,S$612.40,74.0,3.0,Aisha Rahman,NaN,612.400
1,raffles place,01/05/2025,AM,318.75,39.0,2.0,aisha rahman,NaN,318.750
2,RAFFLES PLACE,02/05/2025,morning,-999,55.0,NaN,AISHA RAHMAN,NaN,454.475
3,Raffles Pl.,02/05/2025,Midday,487.10,58.0,2.0,Aisha Rahman,NaN,487.100
4,Marina Bay,01/05/2025,Lunch,"S$1,204.60",141.0,5.0,Priya Nair,aircon down,1204.600
5,marina bay,01/05/2025,midday,296.55,36.0,2.0,priya nair,NaN,296.550
6,Marina Bay,02/05/2025,Evening,,21.0,1.0,Priya Nair,NaN,454.475
7,Marina Bay,02/05/2025,PM,58000,112.0,4.0,Priya Nair,NaN,454.475
8,Tampines Mall,01/05/2025,evening,S$734.20,88.0,4.0,wei ming tan,NaN,734.200
9,tampines mall,02/05/2025,Morning,421.85,51.0,2.0,Wei Ming Tan,NaN,421.850


In [26]:
#5-Standardise `outlet` to the four real café names, and `daypart` to Morning / Midday / Evening.

# Outlet: fix case and spaces first

key = clean["outlet"].str.strip().str.lower()
display(key.unique())

outlet_map = {
    "raffles place": "Raffles Place",
    "raffles pl.": "Raffles Place",
    "marina bay": "Marina Bay",
    "tampines mall": "Tampines Mall",
    "holland village": "Holland Village",
    "holland v": "Holland Village",
}
clean["outlet_name"] = key.map(outlet_map)

# Daypart:

daypart_map = {
    "morning": "Morning", "am": "Morning",
    "midday": "Midday", "lunch": "Midday",
    "evening": "Evening", "pm": "Evening",
}
clean["daypart"] = clean["daypart"].str.strip().str.lower().map(daypart_map)

display(clean)


array(['raffles place', 'raffles pl.', 'marina bay', 'tampines mall',
       'holland village', 'holland v'], dtype=object)

,outlet,date_text,daypart,revenue_raw,tickets,staff_on_shift,manager,notes,revenue_sgd,outlet_name
0,Raffles Place,01/05/2025,Morning,S$612.40,74.0,3.0,Aisha Rahman,NaN,612.400,Raffles Place
1,raffles place,01/05/2025,Morning,318.75,39.0,2.0,aisha rahman,NaN,318.750,Raffles Place
2,RAFFLES PLACE,02/05/2025,Morning,-999,55.0,NaN,AISHA RAHMAN,NaN,454.475,Raffles Place
3,Raffles Pl.,02/05/2025,Midday,487.10,58.0,2.0,Aisha Rahman,NaN,487.100,Raffles Place
4,Marina Bay,01/05/2025,Midday,"S$1,204.60",141.0,5.0,Priya Nair,aircon down,1204.600,Marina Bay
5,marina bay,01/05/2025,Midday,296.55,36.0,2.0,priya nair,NaN,296.550,Marina Bay
6,Marina Bay,02/05/2025,Evening,,21.0,1.0,Priya Nair,NaN,454.475,Marina Bay
7,Marina Bay,02/05/2025,Evening,58000,112.0,4.0,Priya Nair,NaN,454.475,Marina Bay
8,Tampines Mall,01/05/2025,Evening,S$734.20,88.0,4.0,wei ming tan,NaN,734.200,Tampines Mall
9,tampines mall,02/05/2025,Morning,421.85,51.0,2.0,Wei Ming Tan,NaN,421.850,Tampines Mall


In [27]:
#6- Verify: no impossible values left, and `outlet` has exactly four values.

assert clean["revenue_sgd"].between(0, 20000).all()
assert (clean["tickets"] > 0).all()
assert clean["outlet_name"].nunique() == 4
assert clean["daypart"].isin(["Morning", "Midday", "Evening"]).all()
print("All checks passed ✅")

All checks passed ✅


### Challenge 3: Transform & Summarise

**Tasks:**
1. Convert `date_text` to a real datetime column called `date`. Careful: `01/05/2025` is 1 May here.
2. Clean the `manager` column so each manager's name appears exactly once, in Title Case.
3. Add a `shift_size` column with `pd.cut`: 0–350 `"Quiet"`, 350–700 `"Normal"`, 700–1500 `"Busy"`.
4. Build a summary table: one row per café, with total revenue, number of shifts and average revenue
   per shift, sorted by total revenue.

In [28]:
#1-Convert `date_text` to a real datetime column called `date`. Careful: `01/05/2025` is 1 May here.

clean["date"] = pd.to_datetime(clean["date_text"], format="%d/%m/%Y")
display(clean)


,outlet,date_text,daypart,revenue_raw,tickets,staff_on_shift,manager,notes,revenue_sgd,outlet_name,date
0,Raffles Place,01/05/2025,Morning,S$612.40,74.0,3.0,Aisha Rahman,NaN,612.400,Raffles Place,2025-05-01
1,raffles place,01/05/2025,Morning,318.75,39.0,2.0,aisha rahman,NaN,318.750,Raffles Place,2025-05-01
2,RAFFLES PLACE,02/05/2025,Morning,-999,55.0,NaN,AISHA RAHMAN,NaN,454.475,Raffles Place,2025-05-02
3,Raffles Pl.,02/05/2025,Midday,487.10,58.0,2.0,Aisha Rahman,NaN,487.100,Raffles Place,2025-05-02
4,Marina Bay,01/05/2025,Midday,"S$1,204.60",141.0,5.0,Priya Nair,aircon down,1204.600,Marina Bay,2025-05-01
5,marina bay,01/05/2025,Midday,296.55,36.0,2.0,priya nair,NaN,296.550,Marina Bay,2025-05-01
6,Marina Bay,02/05/2025,Evening,,21.0,1.0,Priya Nair,NaN,454.475,Marina Bay,2025-05-02
7,Marina Bay,02/05/2025,Evening,58000,112.0,4.0,Priya Nair,NaN,454.475,Marina Bay,2025-05-02
8,Tampines Mall,01/05/2025,Evening,S$734.20,88.0,4.0,wei ming tan,NaN,734.200,Tampines Mall,2025-05-01
9,tampines mall,02/05/2025,Morning,421.85,51.0,2.0,Wei Ming Tan,NaN,421.850,Tampines Mall,2025-05-02


In [29]:
#2-Clean the `manager` column so each manager's name appears exactly once, in Title Case.
clean["manager"] = clean["manager"].str.split().str.join(" ").str.title()
display(clean)

,outlet,date_text,daypart,revenue_raw,tickets,staff_on_shift,manager,notes,revenue_sgd,outlet_name,date
0,Raffles Place,01/05/2025,Morning,S$612.40,74.0,3.0,Aisha Rahman,NaN,612.400,Raffles Place,2025-05-01
1,raffles place,01/05/2025,Morning,318.75,39.0,2.0,Aisha Rahman,NaN,318.750,Raffles Place,2025-05-01
2,RAFFLES PLACE,02/05/2025,Morning,-999,55.0,NaN,Aisha Rahman,NaN,454.475,Raffles Place,2025-05-02
3,Raffles Pl.,02/05/2025,Midday,487.10,58.0,2.0,Aisha Rahman,NaN,487.100,Raffles Place,2025-05-02
4,Marina Bay,01/05/2025,Midday,"S$1,204.60",141.0,5.0,Priya Nair,aircon down,1204.600,Marina Bay,2025-05-01
5,marina bay,01/05/2025,Midday,296.55,36.0,2.0,Priya Nair,NaN,296.550,Marina Bay,2025-05-01
6,Marina Bay,02/05/2025,Evening,,21.0,1.0,Priya Nair,NaN,454.475,Marina Bay,2025-05-02
7,Marina Bay,02/05/2025,Evening,58000,112.0,4.0,Priya Nair,NaN,454.475,Marina Bay,2025-05-02
8,Tampines Mall,01/05/2025,Evening,S$734.20,88.0,4.0,Wei Ming Tan,NaN,734.200,Tampines Mall,2025-05-01
9,tampines mall,02/05/2025,Morning,421.85,51.0,2.0,Wei Ming Tan,NaN,421.850,Tampines Mall,2025-05-02


In [30]:
#3- Add a `shift_size` column with `pd.cut`: 0–350 `"Quiet"`, 350–700 `"Normal"`, 700–1500 `"Busy"`.
bins   = [0, 350, 700, 1500]
labels = ["Quiet", "Normal", "Busy"]

clean["shift_size"] = pd.cut(clean["revenue_sgd"], bins=bins, labels=labels)
display(clean)

,outlet,date_text,daypart,revenue_raw,tickets,staff_on_shift,manager,notes,revenue_sgd,outlet_name,date,shift_size
0,Raffles Place,01/05/2025,Morning,S$612.40,74.0,3.0,Aisha Rahman,NaN,612.400,Raffles Place,2025-05-01,Normal
1,raffles place,01/05/2025,Morning,318.75,39.0,2.0,Aisha Rahman,NaN,318.750,Raffles Place,2025-05-01,Quiet
2,RAFFLES PLACE,02/05/2025,Morning,-999,55.0,NaN,Aisha Rahman,NaN,454.475,Raffles Place,2025-05-02,Normal
3,Raffles Pl.,02/05/2025,Midday,487.10,58.0,2.0,Aisha Rahman,NaN,487.100,Raffles Place,2025-05-02,Normal
4,Marina Bay,01/05/2025,Midday,"S$1,204.60",141.0,5.0,Priya Nair,aircon down,1204.600,Marina Bay,2025-05-01,Busy
5,marina bay,01/05/2025,Midday,296.55,36.0,2.0,Priya Nair,NaN,296.550,Marina Bay,2025-05-01,Quiet
6,Marina Bay,02/05/2025,Evening,,21.0,1.0,Priya Nair,NaN,454.475,Marina Bay,2025-05-02,Normal
7,Marina Bay,02/05/2025,Evening,58000,112.0,4.0,Priya Nair,NaN,454.475,Marina Bay,2025-05-02,Normal
8,Tampines Mall,01/05/2025,Evening,S$734.20,88.0,4.0,Wei Ming Tan,NaN,734.200,Tampines Mall,2025-05-01,Busy
9,tampines mall,02/05/2025,Morning,421.85,51.0,2.0,Wei Ming Tan,NaN,421.850,Tampines Mall,2025-05-02,Normal


In [31]:
#4-Build a summary table: one row per café, with total revenue, number of shifts and average revenue
summary = (
    clean.groupby("outlet_name")
    .agg(
        revenue=("revenue_sgd", "sum"),
        shifts=("revenue_sgd", "size"),
        avg_shift=("revenue_sgd", "mean"),
    )
    .round(2)
    .sort_values("revenue", ascending=False)
)
display(summary)

,revenue,shifts,avg_shift
outlet_name,,,
Marina Bay,2410.10,4,602.52
Raffles Place,1872.72,4,468.18
Tampines Mall,1156.05,2,578.03
Holland Village,844.38,2,422.19


🏆 Stretch Challenge (optional)
Do the same audit on the real file, data/cafe_june_raw.csv, but for one outlet only:

Do the same audit on the **real** file, `data/cafe_june_raw.csv`, but for one outlet only:

1. Filter to Marina Bay (all three spellings) *before* cleaning. How would you do that safely?
2. Clean it and compute its total June revenue.
3. Export the result to CSV, reload it, and confirm the total and the row count survive the trip.
4. What comes back as a different **type** than it went out as, and why?

In [32]:
# MY ANSWER #

import pandas as pd
import numpy as np
raw = pd.read_csv("../data/cafe_june_raw.csv")

#1-Filter to Marina Bay (all three spellings) *before* cleaning. How would you do that safely?
is_marina = raw["outlet"].str.strip().str.lower() == "marina bay"
marina = raw[is_marina].copy()
display(marina["outlet"].unique())

#2-Clean it and compute its total June revenue.

def clean_revenue(s, low=0, high=20000):
    """Text revenue → number, with impossible values set to NaN."""
    num = pd.to_numeric(s.str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce")
    return num.mask((num < low) | (num > high))

daypart_map = {
    "morning": "Morning", "am": "Morning",
    "midday": "Midday", "lunch": "Midday",
    "evening": "Evening", "pm": "Evening",
}

marina = marina.drop_duplicates()

# NEW: standardise daypart FIRST, so the groups are right
marina["daypart"] = marina["daypart"].str.strip().str.lower().map(daypart_map)

# Mask impossible values
marina["revenue_sgd"] = clean_revenue(marina["revenue_raw"])
marina["tickets"] = marina["tickets"].mask(marina["tickets"] <= 0)

# NEW: fill with each DAYPART's median, not one median for everything
by_daypart = marina.groupby("daypart")
marina["revenue_sgd"] = marina["revenue_sgd"].fillna(by_daypart["revenue_sgd"].transform("median"))
marina["tickets"] = marina["tickets"].fillna(by_daypart["tickets"].transform("median")).astype(int)

marina["date"] = pd.to_datetime(marina["date_text"], format="%d/%m/%Y")

# Verify
display(marina["daypart"].value_counts())                    # expect 30 / 30 / 30
display(marina[["revenue_sgd", "tickets"]].isna().sum())     # expect 0 / 0
print(f"Marina Bay, June 2025: ${marina['revenue_sgd'].sum():,.2f} across {len(marina)} shifts")

#3-Export the result to CSV, reload it, and confirm the total and the row count survive the trip.

out = marina[["date", "daypart", "tickets", "revenue_sgd"]]
out.to_csv("../data/marina_june.csv", index=False)
back = pd.read_csv("../data/marina_june.csv")

print(f"rows: {len(out)} → {len(back)}")
print(f"total: ${out['revenue_sgd'].sum():,.2f} → ${back['revenue_sgd'].sum():,.2f}")


#Imputation choice. I filled missing revenue and tickets with the median of the same daypart at Marina Bay, not one median for the whole café. A single median ($265.64) put morning shifts at evening-level revenue. Filling by daypart uses $565.17 for a morning and $167.87 for an evening. The total moves from $32,058.17 to $32,657.24, which is about $600 that depends only on how I filled two shifts. Rule: fill from the most specific group that still has enough rows.

array(['Marina Bay', 'marina bay', 'Marina Bay '], dtype=object)

Evening    30
Midday     30
Morning    30
Name: daypart, dtype: int64

revenue_sgd    0
tickets        0
dtype: int64

Marina Bay, June 2025: $32,657.24 across 90 shifts
rows: 90 → 90
total: $32,657.24 → $32,657.24


In [33]:
#4 - What comes back as a different **type** than it went out as, and why?
display(out.dtypes) 
display(back.dtypes)

#date: datetime64 --> object
#Why: a CSV is just plain text. It saves what a value looks like (2025-06-01), not what type it is. When pandas reads it back, it guesses each column’s type from the text. 
#Numbers are easy to recognise, but pandas doesn’t try to turn text into dates unless you tell it to.


date           datetime64[ns]
daypart                object
tickets                 int64
revenue_sgd           float64
dtype: object

date            object
daypart         object
tickets          int64
revenue_sgd    float64
dtype: object

💬 Reflection (5 min)
In 2–3 sentences:

In class, the raw June file claimed $267,987 and the truth was $174,753 — and a single mis-keyed shift caused most of the gap. Nothing errored, and nothing looked wrong. What one check will you run from now on, first, on every file you are given? Be specific enough to write it as a line of code.

***Answer*** In class, the $93k gap came from one mis-keyed shift that was too big, not missing. Neither .isna() nor .describe() could catch it, because the revenue column was stored as text and silently left out. Converting first and then looking at the extremes would have shown a max dozens of times larger than the median, which is the mistake in a single glance. 

**Single line of code to check:**
pd.to_numeric(df["revenue_raw"].str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce").describe()

Read min and max against the median before trusting any total.